# Semana 8 · Sistemas distribuidos

**Meta:** identificar latencia, fallos parciales y consistencia; localizar puntos de un monolito candidatos a distribuirse.

| Reto | Trabajo | Aceptación |
|---|---|---|
| 1 · Latencia | Agrupar consultas a un backend | Mismos datos y orden; ≤100 ms simulados por lista |
| 2 · Fallos y consistencia | Reparar la sincronización entre nodos | El nodo sano sigue operando y las copias se recuperan sin duplicados |
| 3 · Monolito modular | Aislar sesión y consulta de viajes | Pruebas aprobadas y candidatos marcados en el código |

**Reglas:** edita solo lo comprendido entre `INICIO ZONA EDITABLE` y `FIN ZONA EDITABLE`. Ejecuta la celda completa del reto: verás **PASA / NO PASA** por caso y un resumen.


## Reto 1 · Latencia

El lector espera una consulta antes de iniciar la siguiente. En **N+1**, una consulta obtiene una lista y luego se hacen N accesos relacionados; aquí repararás esos accesos. Conserva valores, orden y repeticiones con **máximo una llamada al backend** por lista; si está vacía, ninguna.

Adaptación de `id_loader` y pruebas de agrupación/caché de **aiodataloader**
. Se añadió el lector defectuoso y un costo de 100 ms simulados por llamada.


In [1]:
# Preparación y soporte: NO EDITAR.
import asyncio
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("aiodataloader") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "aiodataloader==0.4.3"], check=True)
from aiodataloader import DataLoader

# Adaptación de id_loader(), del archivo de pruebas original.
def id_loader(resolve=None):
    llamadas = []
    async def default_resolve(keys):
        return keys
    async def fn(keys):
        llamadas.append(list(keys))
        return await (resolve or default_resolve)(keys)
    return DataLoader(fn), llamadas

# INICIO ZONA EDITABLE
async def consultar(claves, loader):
    # Devuelve los valores del backend en el orden pedido, con repeticiones.
    # load(clave): una clave; load_many(claves): varias claves.
    # No modifiques claves, loader, caché ni métricas. No inventes valores.
    if not claves:
        return []
    return await loader.load_many(claves)
# FIN ZONA EDITABLE

# PRUEBAS: NO EDITAR. Cada llamada al backend representa 100 ms simulados.
async def _caso(claves, tabla=None):
    async def resolver(keys):
        return [tabla.get(key) for key in keys]
    loader, llamadas = id_loader(resolver if tabla is not None else None)
    entrada = list(claves)
    valores = await consultar(entrada, loader)
    esperado = list(claves) if tabla is None else [tabla.get(k) for k in claves]
    ms = 100 * len(llamadas)
    detalle = f"{len(llamadas)} llamada(s), {ms} ms simulados, valores={valores}"
    assert valores == esperado, "Valores/orden/repeticiones incorrectos. " + detalle
    assert entrada == claves, "Cambiaste la entrada."
    assert len(llamadas) <= (1 if claves else 0), "No agrupaste las solicitudes. " + detalle
    assert ms <= 100, "Excediste el presupuesto de 100 ms. " + detalle
    if claves:
        assert set(llamadas[0]) == set(claves), "No consultaste todas las claves."
    return detalle

async def _evaluar():
    casos = [
        ("dos claves — prueba original adaptada", [1, 2], None),
        ("repetidos — prueba original adaptada", [1, 1, 2], None),
        ("conserva el orden", [9, 2, 6, 1], {1: "Lima", 2: "Cusco", 9: "Abancay"}),
        ("datos distintos del identificador", ["B", "A", "B"], {"A": 17, "B": 43}),
        ("lista vacía — prueba original adaptada", [], None),
        ("una sola clave", [0], None),
    ]
    resultados = []
    for nombre, claves, tabla in casos:
        try:
            detalle = await _caso(claves, tabla)
            pasa = True
        except Exception as error:
            pasa, detalle = False, f"{type(error).__name__}: {error}"
        resultados.append({"caso": nombre, "pasa": pasa, "detalle": detalle})
        print(f"{'PASA' if pasa else 'NO PASA'} | {nombre} | {detalle}")
    print(f"RETO 1: {sum(r['pasa'] for r in resultados)}/{len(resultados)} pruebas")
    return resultados

_resultados = await _evaluar()


NO PASA | dos claves — prueba original adaptada | AssertionError: No agrupaste las solicitudes. 2 llamada(s), 200 ms simulados, valores=[1, 2]
NO PASA | repetidos — prueba original adaptada | AssertionError: No agrupaste las solicitudes. 2 llamada(s), 200 ms simulados, valores=[1, 1, 2]
NO PASA | conserva el orden | AssertionError: No agrupaste las solicitudes. 4 llamada(s), 400 ms simulados, valores=['Abancay', 'Cusco', None, 'Lima']
NO PASA | datos distintos del identificador | AssertionError: No agrupaste las solicitudes. 2 llamada(s), 200 ms simulados, valores=[43, 17, 43]
PASA | lista vacía — prueba original adaptada | 0 llamada(s), 0 ms simulados, valores=[]
PASA | una sola clave | 1 llamada(s), 100 ms simulados, valores=[0]
RETO 1: 2/6 pruebas


## Reto 2 · Fallos parciales y consistencia

Los nodos guardan mensajes. Un enlace puede fallar aunque los otros sigan funcionando; una copia queda atrasada. Repara la sincronización para conservar lo pendiente, continuar con los vecinos sanos y recuperar las copias al restablecer el enlace y ejecutar nuevas rondas.

Adaptación del laboratorio y del ejemplo Python **broadcast de Maelstrom**. RPC sustituido por llamadas locales con cortes controlados; pruebas de aula añadidas. Reintentar un mensaje no debe duplicarlo.


In [ ]:
# Soporte: NO EDITAR. RPC local con fallos controlados, sin llamadas de red.
class Red:
    def __init__(self):
        self.nodos, self.caidos, self.perder_respuesta = {}, set(), set()
        self.ack_invalido, self.envios = set(), []

    def rpc(self, origen, destino, cuerpo):
        self.envios.append((origen, destino, dict(cuerpo)))
        if destino in self.caidos:
            raise ConnectionError(f"Sin conexión con {destino}")
        respuesta = self.nodos[destino].recibir(cuerpo)
        if (origen, destino) in self.perder_respuesta:
            self.perder_respuesta.remove((origen, destino))
            raise TimeoutError("El receptor guardó; la respuesta no llegó")
        if destino in self.ack_invalido:
            return {"type": "error"}
        return respuesta

class Nodo:
    def __init__(self, nombre, red):
        self.nombre, self.red = nombre, red
        self.mensajes, self.confirmados = set(), {}
        red.nodos[nombre] = self

    def broadcast(self, mensaje):
        self.mensajes.add(mensaje)
        return {"type": "broadcast_ok"}

    def recibir(self, cuerpo):
        self.mensajes.update(cuerpo["messages"])
        return {"type": "broadcast_many_ok"}

    def leer(self):
        return set(self.mensajes)

    # INICIO ZONA EDITABLE
    def sincronizar(self):
        # Una ronda: intenta los vecinos aunque falle uno; no esperes en bucle.
        # Conserva los mensajes locales. Un timeout no demuestra que no llegaron.
        # Confirma SOLO lo que tenga respuesta broadcast_many_ok.
        for vecino in self.red.nodos:
            if vecino == self.nombre:
                continue
            confirmados = self.confirmados.setdefault(vecino, set())
            pendientes = self.mensajes - confirmados
            if not pendientes:
                continue
            try:
                respuesta = self.red.rpc(self.nombre, vecino, {
                    "type": "broadcast_many", "messages": list(pendientes)})
            except (ConnectionError, TimeoutError):
                continue
            if respuesta.get("type") == "broadcast_many_ok":
                confirmados.update(pendientes)
    # FIN ZONA EDITABLE

# PRUEBAS: NO EDITAR. La recuperación se comprueba ejecutando nuevas rondas.
def _crear(*nombres):
    red = Red()
    return red, [Nodo(nombre, red) for nombre in nombres]

def _normal():
    red, (a, b) = _crear("A", "B")
    a.broadcast(7)
    a.sincronizar()
    assert a.leer() == b.leer() == {7}, "La copia no recibió el mensaje."
    return "A={7}, B={7}"

def _local_con_corte():
    red, (a, b) = _crear("A", "B")
    red.caidos.add("B")
    assert a.broadcast(7) == {"type": "broadcast_ok"}
    assert a.leer() == {7} and b.leer() == set()
    return "A acepta el mensaje; B sigue atrasado"

def _otro_vecino():
    red, (a, b, c) = _crear("A", "B", "C")
    red.caidos.add("B")
    a.broadcast(7)
    a.sincronizar()
    assert a.leer() == c.leer() == {7} and not b.leer()
    return "falló B; C sí recibió y A no perdió su dato"

def _pendiente():
    red, (a, b) = _crear("A", "B")
    red.caidos.add("B")
    a.broadcast(7)
    a.sincronizar()
    assert a.confirmados.get("B", set()) == set(), "Confirmaste sin respuesta."
    assert a.leer() == {7} and not b.leer()
    return "sin confirmación: el mensaje queda pendiente para B"

def _recuperacion():
    red, (a, b) = _crear("A", "B")
    a.broadcast(7)
    red.caidos.add("B")
    a.sincronizar()
    red.caidos.clear()
    a.sincronizar()
    assert a.leer() == b.leer() == {7}, "La copia no se recuperó tras el corte."
    return "tras recuperar el enlace y reintentar: ambas copias={7}"

def _respuesta_perdida():
    red, (a, b) = _crear("A", "B")
    red.perder_respuesta.add(("A", "B"))
    a.broadcast(7)
    a.sincronizar()
    assert b.leer() == {7}, "El escenario sí entrega antes del timeout."
    assert not a.confirmados.get("B", set()), "El emisor no recibió confirmación."
    a.sincronizar()
    assert a.leer() == b.leer() == {7} and len(red.envios) == 2
    return "se perdió la respuesta; reintentar confirmó sin duplicar el mensaje"

def _ack_invalido():
    red, (a, b) = _crear("A", "B")
    red.ack_invalido.add("B")
    a.broadcast(7)
    a.sincronizar()
    assert not a.confirmados.get("B", set()), "Una respuesta error no es confirmación."
    red.ack_invalido.clear()
    a.sincronizar()
    assert a.confirmados["B"] == {7}
    return "se confirma solo después de broadcast_many_ok"

def _sin_reenvio():
    red, (a, b) = _crear("A", "B")
    a.broadcast(7)
    a.sincronizar()
    llamadas = len(red.envios)
    a.sincronizar()
    assert len(red.envios) == llamadas and b.leer() == {7}, "Reenviaste lo ya confirmado."
    return "ningún envío nuevo si no hay pendientes"

def _convergencia():
    red, (a, b, c) = _crear("A", "B", "C")
    a.broadcast(7)
    b.broadcast(11)
    red.caidos.add("C")
    a.sincronizar()
    b.sincronizar()
    red.caidos.clear()
    for _ in range(3):
        for nodo in (a, b, c):
            nodo.sincronizar()
    assert all(n.leer() == {7, 11} for n in (a, b, c)), "No convergieron las copias."
    return "tras el corte y las rondas de recuperación: A=B=C={7, 11}"

_resultados = []
for nombre, caso in [
    ("copiar sin fallo", _normal),
    ("aceptar localmente durante el corte", _local_con_corte),
    ("un vecino caído no bloquea al sano", _otro_vecino),
    ("sin respuesta no confirmar", _pendiente),
    ("recuperar una copia atrasada", _recuperacion),
    ("reintentar una respuesta perdida", _respuesta_perdida),
    ("respuesta incorrecta no confirma", _ack_invalido),
    ("no reenviar lo confirmado", _sin_reenvio),
    ("convergencia de tres copias", _convergencia),
]:
    try:
        detalle, pasa = caso(), True
    except Exception as error:
        pasa, detalle = False, f"{type(error).__name__}: {error}"
    _resultados.append({"caso": nombre, "pasa": pasa, "detalle": detalle})
    print(f"{'PASA' if pasa else 'NO PASA'} | {nombre} | {detalle}")
print(f"RETO 2: {sum(r['pasa'] for r in _resultados)}/{len(_resultados)} pruebas")


## Reto 3 · Monolito modular

El servicio de viajes llama directamente a una sesión global y a un acceso global a datos. Reemplaza esos accesos por los colaboradores que recibe, conserva las reglas de acceso y marca en el código los puntos candidatos a convertirse en integraciones externas.

Adaptación de **Trip Service Kata**, de Sandro Mancuso [5]. Se conservan sus reglas y llamadas legadas; se añaden colaboradores locales y pruebas para practicar la separación de dependencias.


In [ ]:
# Soporte: NO EDITAR. Clases y reglas adaptadas de Trip Service Kata.
class UserNotLoggedInException(Exception):
    pass

class User:
    def __init__(self):
        self.trips, self.friends = [], []
    def addFriend(self, user):
        self.friends.append(user)
    def addTrip(self, trip):
        self.trips.append(trip)
    def getFriends(self):
        return self.friends

# Dependencias globales del código legado: no deben usarse en los tests.
def _getLoggedUser():
    raise RuntimeError("Se usó la sesión global en vez del colaborador recibido")

def _findTripsByUser(user):
    raise RuntimeError("Se usó el DAO global en vez del colaborador recibido")

class Sesion:
    def __init__(self, usuario, disponible=True):
        self.usuario, self.disponible, self.llamadas = usuario, disponible, 0
    def getLoggedUser(self):
        self.llamadas += 1
        if not self.disponible:
            raise ConnectionError("Identidad no disponible")
        return self.usuario

class Viajes:
    def __init__(self, datos, disponible=True):
        self.datos, self.disponible, self.llamadas = list(datos), disponible, []
    def findTripsByUser(self, user):
        self.llamadas.append(user)
        if not self.disponible:
            raise ConnectionError("Consulta de viajes no disponible")
        return list(self.datos)

# INICIO ZONA EDITABLE
class TripService:
    def __init__(self, sesion, viajes):
        self.sesion, self.viajes = sesion, viajes

    def getTripsByUser(self, user):
        # Sustituye las dependencias globales por los colaboradores recibidos.
        # Conserva: sin sesión => excepción; sin amistad => []; con amistad => viajes.
        # Si identidad falla, no inventes autorización ni consultes viajes.
        # Marca con # CANDIDATO los accesos que podrían pasar a un servicio externo.
        loggedUser = self.sesion.getLoggedUser()  # CANDIDATO: proveedor de identidad externo.
        if loggedUser:
            if loggedUser in user.getFriends():
                return self.viajes.findTripsByUser(user)  # CANDIDATO: repositorio externo.
            else:
                return []
        else:
            raise UserNotLoggedInException()
# FIN ZONA EDITABLE

# PRUEBAS: NO EDITAR. Sesión y consulta siguen siendo objetos locales.
def _error_esperado(tipo, accion):
    try:
        accion()
    except tipo:
        return
    raise AssertionError(f"Se esperaba {tipo.__name__}")

def _sin_sesion():
    sesion, viajes = Sesion(None), Viajes(["Cusco"])
    _error_esperado(UserNotLoggedInException,
                    lambda: TripService(sesion, viajes).getTripsByUser(User()))
    assert sesion.llamadas == 1 and not viajes.llamadas
    return "sin login: acceso rechazado, cero consultas de viajes"

def _sin_amistad():
    sesion, viajes = Sesion(User()), Viajes(["Cusco"])
    assert TripService(sesion, viajes).getTripsByUser(User()) == []
    assert sesion.llamadas == 1 and not viajes.llamadas
    return "sin amistad: lista vacía, cero consultas de viajes"

def _amigo(datos):
    visitante, propietario = User(), User()
    propietario.addFriend(visitante)
    sesion, viajes = Sesion(visitante), Viajes(datos)
    salida = TripService(sesion, viajes).getTripsByUser(propietario)
    assert salida == datos and viajes.llamadas == [propietario]
    assert sesion.llamadas == 1
    return f"acceso autorizado: {salida}; una consulta al colaborador"

def _direccion_amistad():
    visitante, propietario = User(), User()
    visitante.addFriend(propietario)
    viajes = Viajes(["Cusco"])
    assert TripService(Sesion(visitante), viajes).getTripsByUser(propietario) == []
    assert not viajes.llamadas, "La amistad debe estar en el perfil del propietario."
    return "se conserva la regla de la kata, sin invertir la relación"

def _identidad_caida():
    visitante, propietario = User(), User()
    propietario.addFriend(visitante)
    sesion, viajes = Sesion(visitante, disponible=False), Viajes(["Cusco"])
    _error_esperado(ConnectionError,
                    lambda: TripService(sesion, viajes).getTripsByUser(propietario))
    assert not viajes.llamadas and sesion.llamadas == 1
    return "identidad caída: no se autoriza y no se consulta el repositorio"

def _datos_caidos():
    visitante, propietario = User(), User()
    propietario.addFriend(visitante)
    viajes = Viajes([], disponible=False)
    _error_esperado(ConnectionError,
                    lambda: TripService(Sesion(visitante), viajes).getTripsByUser(propietario))
    assert viajes.llamadas == [propietario]
    return "se informa el fallo de consulta, no se inventa una lista válida"

def _instancias_independientes():
    visitante, propietario = User(), User()
    propietario.addFriend(visitante)
    s1, s2 = Sesion(visitante), Sesion(visitante)
    v1, v2 = Viajes(["Abancay"]), Viajes(["Lima", "Cusco"])
    a, b = TripService(s1, v1), TripService(s2, v2)
    assert a.getTripsByUser(propietario) == ["Abancay"]
    assert b.getTripsByUser(propietario) == ["Lima", "Cusco"]
    assert s1.llamadas == s2.llamadas == 1
    assert v1.llamadas == v2.llamadas == [propietario]
    return "cada servicio usa sus colaboradores, sin estado global compartido"

_resultados = []
for nombre, caso in [
    ("usuario sin login", _sin_sesion),
    ("usuario no amigo", _sin_amistad),
    ("amigo con viajes", lambda: _amigo(["Cusco", "Lima"])),
    ("amigo sin viajes", lambda: _amigo([])),
    ("dirección de la amistad", _direccion_amistad),
    ("proveedor de identidad caído", _identidad_caida),
    ("consulta de viajes caída", _datos_caidos),
    ("dependencias sustituibles", _instancias_independientes),
]:
    try:
        detalle, pasa = caso(), True
    except Exception as error:
        pasa, detalle = False, f"{type(error).__name__}: {error}"
    _resultados.append({"caso": nombre, "pasa": pasa, "detalle": detalle})
    print(f"{'PASA' if pasa else 'NO PASA'} | {nombre} | {detalle}")
print(f"RETO 3: {sum(r['pasa'] for r in _resultados)}/{len(_resultados)} pruebas")


## Sources

[5] https://github.com/sandromancuso/trip-service-kata/blob/master/python/tripservice_original.py
[8] https://github.com/jepsen-io/maelstrom/blob/main/demo/python/broadcast.py
[9] https://github.com/jepsen-io/maelstrom/blob/main/doc/03-broadcast/01-broadcast.md
[10] https://github.com/syrusakbary/aiodataloader/blob/master/test_aiodataloader.py
